# Generación de audio sintético Shipibo-Konibo (Tacotron 2 + HiFi-GAN)

Sintetiza el texto no pareado (`shp_apto_tts.txt`) con el Tacotron 2 y el HiFi-GAN entrenados por
Menéndez y Erasmo, y guarda los wavs en Drive como **audio generado**, con un `manifest.csv`
que los empareja con su texto.

**Reproducible:** repositorios en commits fijos, una semilla por frase derivada de su id,
semilla fija en el denoiser, y un `config.json` con versiones, parámetros y huellas SHA-256
del texto y de los dos modelos.

**Reanudable:** si Colab se desconecta, vuelve a ejecutar todo — salta lo ya generado. Como
la semilla depende del id y no del orden, retomar da exactamente el mismo audio.

Ejecuta las celdas en orden. La celda 7 es una prueba corta: **escúchala antes de lanzar la 8.**

## 1. Parámetros

In [ ]:
#@title Parámetros { display-mode: "form" }
#@markdown **Entrada por ID de Drive (recomendado).** Pega el ID o el enlace de compartir de cada
#@markdown archivo. Deben estar compartidos como *Cualquier persona con el enlace (Lector)*.
#@markdown Da igual cuál pongas en cuál: el tipo de modelo se detecta por su contenido.
TACOTRON_ID = ""  #@param {type:"string"}
HIFIGAN_ID = ""  #@param {type:"string"}
TEXTO_ID = ""  #@param {type:"string"}

#@markdown **Salida en Drive** (tiene que ser Drive: en `/content` se pierde al desconectarse)
SALIDA = "/content/drive/MyDrive/TESIS/audio_generado"  #@param {type:"string"}

#@markdown **Alternativa: carpeta en Drive.** Solo se usa si los tres IDs están vacíos.
DRIVE_MODELOS = "/content/drive/MyDrive/TESIS/modelo_tts"  #@param {type:"string"}
#@markdown Nombres de archivo dentro de `DRIVE_MODELOS`. Vacío = detección automática.
TACOTRON_CKPT = ""  #@param {type:"string"}
HIFIGAN_CKPT = ""  #@param {type:"string"}
TEXTO = "shp_apto_tts.txt"  #@param {type:"string"}

#@markdown **Reproducibilidad**
SEED = 1234  #@param {type:"integer"}

#@markdown **Texto** (SK001, con el que se entrenó el modelo, va de 2 a 20 palabras por frase)
MAX_PALABRAS = 20  #@param {type:"integer"}
MIN_PALABRAS = 2  #@param {type:"integer"}
#@markdown Cuántas frases generar en esta corrida. 0 = todas.
N_MAX = 0  #@param {type:"integer"}

#@markdown **Síntesis** (valores del notebook de Menéndez y Erasmo)
MAX_DUR_S = 20  #@param {type:"integer"}
STOP_THRESHOLD = 0.5  #@param {type:"number"}
FUERZA_DENOISER = 35.0  #@param {type:"number"}
#@markdown 22050 = salida nativa del modelo. 16000 = mismo formato que el resto del corpus.
OUTPUT_SR = 22050  #@param [22050, 16000] {type:"raw"}

#@markdown **Control de calidad** (solo marca, nunca borra: todo queda en el manifest)
FOCO_MIN = 0.30  #@param {type:"number"}
COBERTURA_MIN = 0.85  #@param {type:"number"}
RATIO_DUR_MIN = 0.4  #@param {type:"number"}
RATIO_DUR_MAX = 2.5  #@param {type:"number"}

# --- fijos ---
TACOTRON_REPO, TACOTRON_COMMIT = "https://github.com/rmcpantoja/tacotron2.git", "0b33108d515c8dfaef4cc6fc1c4691c9c419bfa4"
HIFIGAN_REPO, HIFIGAN_COMMIT = "https://github.com/justinjohn0306/hifi-gan", "11a178893bad3df9a918b405e3cf9c48ee682487"
CPS_SK001 = 10.31          # caracteres/s medidos en SK001: duración esperada de cada frase
MAX_PASOS = int(MAX_DUR_S * 22050 / 256)   # 22050 Hz / hop 256 = 86.13 pasos por segundo

## 2. Drive y GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch, subprocess
if not torch.cuda.is_available():
    raise SystemExit("Sin GPU. Entorno de ejecución > Cambiar tipo de entorno > GPU (T4).")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)

## 3. Dependencias y repositorios

No se reinstala PyTorch: el `torch==1.13.1` del notebook original ya no existe para el Python
actual de Colab. Se usa el de Colab. Ambos repositorios ya son compatibles con PyTorch 2.x
y librosa ≥ 0.10 (verificado).

In [ ]:
!pip install -q unidecode inflect
!pip install -q -U gdown
import os, sys
os.chdir("/content")
for url, commit, carpeta in [(TACOTRON_REPO, TACOTRON_COMMIT, "tacotron2"),
                             (HIFIGAN_REPO, HIFIGAN_COMMIT, "hifi-gan")]:
    if not os.path.isdir(carpeta):
        # sin --recursive: los submódulos de waveglow no se usan en inferencia
        !git clone -q {url} {carpeta}
    !git -C {carpeta} checkout -q {commit}
    print(carpeta, "@", subprocess.run(["git", "-C", carpeta, "rev-parse", "HEAD"],
                                       capture_output=True, text=True).stdout.strip())

# Mismo orden que el notebook original. Ambos repos traen stft.py y audio_processing.py:
# son idénticos byte a byte, así que el orden no cambia el resultado.
for p in ("/content/hifi-gan", "/content/tacotron2"):
    if p not in sys.path:
        sys.path.append(p)

## 4. Núcleo de síntesis

Este es el código que se probó localmente. Se escribe a disco y se importa.

In [ ]:
%%writefile /content/tts_core.py
# ---- nucleo de sintesis: se prueba localmente y se copia tal cual al notebook ----
import csv, hashlib, json, os, random, re, time
import numpy as np
import torch

# ============================ TEXTO ============================
# Caracteres que el Tacotron2 no conoce (text/symbols.py). Sin mapear, se descartan en silencio.
_MAPA = {"—": ", ", "–": ", ", "/": " ",               # guiones largos y barra -> pausa/espacio
         '"': "", "“": "", "”": "", "‘": "", "’": ""}  # comillas -> nada
_CORTE_FUERTE = re.compile(r"(?<=[.!?])\s+")
_CORTE_SUAVE = re.compile(r"(?<=[,:;])\s+")


def normalizar(linea):
    t = linea.strip()
    for a, b in _MAPA.items():
        t = t.replace(a, b)
    # ';' es el token de fin de frase del modelo: uno interno podria cortarlo antes de tiempo
    t = t.rstrip(";").replace(";", ",")
    t = re.sub(r"\s+", " ", t).strip(" ,")
    return t


def _n(t):
    return len(t.split())


def partir(texto, max_pal):
    """Parte en <= max_pal palabras: primero en . ! ?, luego en , : ;, y en ultimo caso por palabras."""
    if _n(texto) <= max_pal:
        return [texto]
    salida = []
    for trozo in _CORTE_FUERTE.split(texto):
        if _n(trozo) <= max_pal:
            salida.append(trozo)
            continue
        actual = ""
        for sub in _CORTE_SUAVE.split(trozo):
            cand = (actual + " " + sub).strip()
            if _n(cand) <= max_pal:
                actual = cand
            else:
                if actual:
                    salida.append(actual)
                actual = sub
                while _n(actual) > max_pal:            # sin puntuacion util: corte por palabras
                    p = actual.split()
                    salida.append(" ".join(p[:max_pal]))
                    actual = " ".join(p[max_pal:])
        if actual:
            salida.append(actual)
    return [s.strip(" ,") for s in salida if s.strip(" ,")]


def construir_items(ruta_txt, max_pal, min_pal):
    """Devuelve [(id, texto_original, texto_tts)] con ids estables ligados a la linea de origen."""
    items = []
    with open(ruta_txt, encoding="utf-8-sig") as f:
        for i, linea in enumerate(f):
            if not linea.strip():
                continue
            base = normalizar(linea)
            for j, trozo in enumerate(partir(base, max_pal)):
                if _n(trozo) >= min_pal:
                    items.append((f"L{i:05d}_{j:02d}", linea.strip(), trozo))
    return items


def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


# ============================ SEMILLAS ============================
def fijar_semilla(s):
    random.seed(s)
    np.random.seed(s % (2 ** 32))
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# ============================ MODELOS ============================
def cargar_tacotron(ruta, device, max_decoder_steps, gate_threshold):
    from hparams import create_hparams
    from model import Tacotron2
    hp = create_hparams()
    hp.sampling_rate = 22050
    hp.max_decoder_steps = max_decoder_steps
    hp.gate_threshold = gate_threshold
    m = Tacotron2(hp).to(device)
    if ruta is not None:
        # weights_only=False: desde torch 2.6 el default es True y rechaza estos checkpoints
        ck = torch.load(ruta, map_location=device, weights_only=False)
        sd = ck["state_dict"] if "state_dict" in ck else ck
        if any("mi." in k for k in sd):
            raise RuntimeError("Checkpoint MMI: no compatible con este Tacotron2.")
        m.load_state_dict(sd)
    m.decoder.max_decoder_steps = max_decoder_steps
    m.decoder.gate_threshold = gate_threshold
    return m.eval(), hp


def cargar_hifigan(ruta, ruta_config, device, seed_denoiser):
    from env import AttrDict
    from models import Generator
    from denoiser import Denoiser
    with open(ruta_config) as f:
        h = AttrDict(json.load(f))
    g = Generator(h).to(device)
    if ruta is not None:
        ck = torch.load(ruta, map_location=device, weights_only=False)
        g.load_state_dict(ck["generator"] if "generator" in ck else ck)
    g.eval()
    g.remove_weight_norm()
    # El Denoiser 'normal' estima su sesgo con torch.randn: sin semilla, cada corrida da otro audio.
    fijar_semilla(seed_denoiser)
    den = Denoiser(g, mode="normal")
    return g, h, den


# ============================ SINTESIS ============================
MAX_WAV_VALUE = 32768.0
MIN_FRAMES = 8          # 8 frames x 256 hop = 2048 muestras: margen sobre las 1024 del STFT


def _qc(align, n_pasos, max_pasos, dur_s, texto, cps):
    a = align.float().cpu().numpy()[0]                   # (T_dec, T_enc)
    foco = float(a.max(axis=1).mean())                   # 1.0 = atencion perfectamente enfocada
    fin = float(a[-1].argmax() / max(a.shape[1] - 1, 1)) # ~1.0 si la atencion llego al final del texto
    esperado = len(texto.replace(" ", "")) / cps          # duracion esperada segun la tasa de SK001
    return dict(pasos=n_pasos, toco_max=int(n_pasos >= max_pasos), foco=round(foco, 3),
                cobertura_fin=round(fin, 3), dur_esperada=round(esperado, 2),
                ratio_dur=round(dur_s / esperado, 3) if esperado > 0 else None)


def estado_qc(q, foco_min, cobertura_min, ratio_rango):
    f = []
    if q.get("muy_corto"):
        f.append("muy_corto")
    if q["toco_max"]:
        f.append("no_se_detuvo")
    if q["foco"] < foco_min:
        f.append("atencion_difusa")
    if q["cobertura_fin"] < cobertura_min:
        f.append("no_llego_al_final")
    r = q["ratio_dur"]
    if r is not None and not (ratio_rango[0] <= r <= ratio_rango[1]):
        f.append("duracion_anomala")
    return "ok" if not f else ",".join(f)


@torch.no_grad()
def sintetizar(texto, seed, tacotron, hifigan, denoiser, device, max_pasos, cps,
               fuerza_denoiser=35.0):
    from text import text_to_sequence
    fijar_semilla(seed)                                  # por frase: reproducible y reanudable
    t = texto if texto.endswith(";") else texto + ";"    # token de fin, igual que el notebook original
    seq = torch.LongTensor(text_to_sequence(t, ["basic_cleaners"]))[None, :].to(device)
    _, mel, _, align = tacotron.inference(seq)
    if mel.shape[-1] < MIN_FRAMES:
        # el decoder se detuvo casi de inmediato (falla de atencion). El STFT del denoiser
        # necesita > 1024 muestras y crashearia la corrida entera: se reporta y se sigue.
        q = _qc(align, mel.shape[-1], max_pasos, mel.shape[-1] * 256 / 22050, texto, cps)
        q["muy_corto"] = 1
        return None, q
    y = hifigan(mel.float()).squeeze() * MAX_WAV_VALUE
    y = denoiser(y.view(1, -1), strength=fuerza_denoiser)[:, 0]
    y = y.cpu().numpy().reshape(-1) / MAX_WAV_VALUE
    pico = float(np.max(np.abs(y))) if y.size else 0.0
    if pico > 0:
        y = y * (10 ** (-1 / 20) / pico)                 # pico a -1 dBFS
    dur = len(y) / 22050
    return y.astype(np.float32), _qc(align, mel.shape[-1], max_pasos, dur, texto, cps)

## 5. Modelos, texto y salida

Con IDs los descarga; si no, los busca en la carpeta de Drive. Los `.zip` se abren solos.

In [ ]:
import sys, importlib, os, re, shutil, zipfile
sys.path.insert(0, "/content")
import tts_core as C
importlib.reload(C)
from pathlib import Path

RAIZ = Path("/content/drive")
EXT_DATOS = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".txt", ".csv", ".json", ".xlsx", ".md", ".ipynb"}
DESCARGAS = Path("/content/modelos")
EXTRAIDOS = Path("/content/modelos_extraidos")
_listado = []


def ver_mydrive():
    if _listado:
        return
    _listado.append(1)
    print("Esto es lo que Colab ve en tu MyDrive. Si NO reconoces estas carpetas,")
    print("montaste otra cuenta de Google: vuelve a la celda 2 y elige la correcta.")
    for f in sorted((RAIZ / "MyDrive").iterdir())[:30]:
        print(f"   {f.name}{'/' if f.is_dir() else ''}")
    print()


def buscar_carpeta(nombre, prof=4):
    """Busca en MyDrive, en accesos directos a carpetas compartidas y en unidades compartidas."""
    hallados = []
    for r in (RAIZ / "MyDrive", RAIZ / ".shortcut-targets-by-id", RAIZ / "Shareddrives"):
        if not r.is_dir():
            continue
        base = len(r.parts)
        for dirpath, dirnames, _ in os.walk(r):
            d = Path(dirpath)
            if len(d.parts) - base >= prof:
                dirnames[:] = []                     # no bajar más: Drive por FUSE es lento
            if d.name.strip().lower() == nombre.strip().lower():
                hallados.append(d)
    return hallados


def reubicar(ruta, param):
    """Devuelve la carpeta; si no está en esa ruta exacta, la busca por nombre en todo Drive."""
    ruta = Path(ruta)
    if ruta.is_dir():
        return ruta
    print(f"No existe {ruta}")
    if not (RAIZ / "MyDrive").is_dir():
        raise SystemExit("Drive no está montado. Ejecuta de nuevo la celda 2 y acepta los permisos.")
    print()
    ver_mydrive()
    print(f"Buscando una carpeta '{ruta.name}' en tu Drive...")
    hallados = buscar_carpeta(ruta.name)
    if len(hallados) != 1:
        raise SystemExit(f"Encontré {len(hallados)} carpetas '{ruta.name}': "
                         f"{[str(h) for h in hallados]}. Corrige {param} en la celda 1.")
    print(f"La encontré en otra ruta y la uso: {hallados[0]}")
    print(f"Para no ver este aviso, corrige {param} en la celda 1.")
    print()
    return hallados[0]


def candidatos_de(f):
    """Rutas a probar como checkpoint. Un .zip puede ser el checkpoint mismo (torch guarda
    en formato zip desde la v1.6) o un comprimido que lo contiene."""
    if f.suffix.lower() != ".zip":
        return [f]
    try:
        zf = zipfile.ZipFile(f)
    except zipfile.BadZipFile:
        return [f]
    with zf:
        nombres = [n for n in zf.namelist() if not n.endswith("/")]
        if any(n.endswith("data.pkl") for n in nombres):
            return [f]                               # checkpoint de torch con extensión .zip
        if len(nombres) > 20 or sum(Path(n).suffix.lower() in EXT_DATOS for n in nombres) > len(nombres) / 2:
            print(f"   {f.name}: es un dataset ({len(nombres)} archivos), no se descomprime")
            return []
        destino = EXTRAIDOS / f.stem
        if not destino.exists():
            tmp = destino.with_name(destino.name + ".tmp")   # si se corta, no queda a medias
            shutil.rmtree(tmp, ignore_errors=True)
            print(f"   descomprimiendo {f.name} ...")
            zf.extractall(tmp)
            tmp.rename(destino)
    return [p for p in sorted(destino.rglob("*")) if p.is_file() and p.stat().st_size > 5e6
            and p.suffix.lower() not in EXT_DATOS]


def clasificar(ruta):
    try:
        ck = torch.load(ruta, map_location="cpu", weights_only=False)
    except Exception:
        return None
    if isinstance(ck, dict) and "generator" in ck:
        return "hifigan"
    sd = ck.get("state_dict", ck) if isinstance(ck, dict) else {}
    if isinstance(sd, dict) and any(k.startswith(("embedding", "encoder", "decoder")) for k in sd):
        return "tacotron"
    return None


candidatos = {"tacotron": [], "hifigan": []}       # (archivo de origen, checkpoint real)


def registrar(f):
    for c in candidatos_de(f):
        tipo = clasificar(c)
        print(f"   {c.name:40s} -> {tipo or 'no es un modelo'}")
        if tipo:
            candidatos[tipo].append((f, c))


def elegir(tipo, manual="", evitar=()):
    c = candidatos[tipo]
    if manual:
        c = [x for x in c if manual in (x[0].name, x[1].name)]
        if not c:
            raise SystemExit(f"'{manual}' no es un modelo {tipo} válido")
    if len(c) > 1:  # p.ej. el HiFi-GAN de superresolución también tiene 'generator'
        c = [x for x in c if not any(e in (x[0].name + x[1].name).lower() for e in evitar)] or c
    if len(c) != 1:
        raise SystemExit(f"{tipo}: {len(c)} candidatos {[x[0].name for x in c]}. "
                         f"Revisa los IDs o los nombres en la celda 1.")
    return c[0][1]


def id_de(valor):
    """Acepta el ID o el enlace de compartir completo."""
    v = valor.strip()
    m = re.search(r"/d/([-\w]{20,})", v) or re.search(r"[?&]id=([-\w]{20,})", v)
    return m.group(1) if m else v


def bajar(valor, que):
    import gdown
    fid = id_de(valor)
    DESCARGAS.mkdir(parents=True, exist_ok=True)
    marca = DESCARGAS / f".{fid}"                    # recuerda el nombre: no re-descarga en la sesión
    if marca.exists() and (DESCARGAS / marca.read_text().strip()).exists():
        p = DESCARGAS / marca.read_text().strip()
        print(f"   {que:11s} ya descargado: {p.name}")
        return p
    print(f"   {que:11s} descargando {fid} ...")
    try:
        out = gdown.download(id=fid, output=str(DESCARGAS) + "/", quiet=True)
    except Exception as e:
        out = None
        print(f"      {type(e).__name__}: {e}")
    if not out or not Path(out).exists():
        raise SystemExit(f"No pude descargar {que} ({fid}). En Drive: clic derecho > Compartir > "
                         f"Acceso general: 'Cualquier persona con el enlace' (Lector).")
    p = Path(out)
    marca.write_text(p.name)
    print(f"   {'':11s} -> {p.name} ({p.stat().st_size / 1e6:.1f} MB)")
    return p


# ------------------------------------------------------------------ entrada
ids = [TACOTRON_ID.strip(), HIFIGAN_ID.strip(), TEXTO_ID.strip()]
if any(ids):
    if not all(ids):
        raise SystemExit("Pon los tres IDs (Tacotron, HiFi-GAN y texto), o déjalos vacíos "
                         "para usar la carpeta DRIVE_MODELOS.")
    print("Descargando por ID:")
    origenes = [bajar(TACOTRON_ID, "Tacotron 2"), bajar(HIFIGAN_ID, "HiFi-GAN")]
    RUTA_TEXTO = bajar(TEXTO_ID, "texto")
    print()
    print("Identificando modelos por su contenido:")
    for f in origenes:
        registrar(f)
    RUTA_TACOTRON = elegir("tacotron")
    RUTA_HIFIGAN = elegir("hifigan")
else:
    carpeta = reubicar(DRIVE_MODELOS, "DRIVE_MODELOS")
    print(f"Contenido de {carpeta}:")
    for f in sorted(carpeta.iterdir()):
        print(f"   {f.name:50s} {f.stat().st_size/1e6:9.1f} MB" if f.is_file() else f"   {f.name}/")
    print()
    print("Buscando modelos:")
    for f in sorted(carpeta.iterdir()):
        if f.is_file() and f.stat().st_size > 5e6 and f.suffix.lower() not in EXT_DATOS:
            registrar(f)
    RUTA_TACOTRON = elegir("tacotron", TACOTRON_CKPT)
    RUTA_HIFIGAN = elegir("hifigan", HIFIGAN_CKPT, evitar=("superres", "32k", "twilight"))
    RUTA_TEXTO = carpeta / TEXTO
    if not RUTA_TEXTO.exists():
        raise SystemExit(f"No encuentro {RUTA_TEXTO.name} en {carpeta}. "
                         f"Hay: {[t.name for t in sorted(carpeta.glob('*.txt'))]}")

# ------------------------------------------------------------------ salida
# La carpeta de salida se crea si falta, pero su carpeta padre tiene que existir:
# así no se crea por error un 'TESIS' nuevo en la raíz, ni en otra cuenta.
salida = Path(SALIDA)
if not salida.parent.is_dir():
    salida = reubicar(salida.parent, "SALIDA") / salida.name
    SALIDA = str(salida)

print()
print(f"Tacotron 2 : {RUTA_TACOTRON}")
print(f"HiFi-GAN   : {RUTA_HIFIGAN}")
print(f"Texto      : {RUTA_TEXTO}")
print(f"Salida     : {SALIDA}")

## 6. Preparar texto y cargar modelos

In [ ]:
import numpy as np, zlib, json, platform, librosa
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

items = C.construir_items(RUTA_TEXTO, MAX_PALABRAS, MIN_PALABRAS)
n_pal = np.array([len(t.split()) for _, _, t in items])
n_lin = sum(1 for l in open(RUTA_TEXTO, encoding="utf-8-sig") if l.strip())
usadas = {i[:6] for i, _, _ in items}                      # 'L00123' = línea de origen
print(f"{len(items)} frases a partir de {len(usadas)} de {n_lin} líneas "
      f"({n_lin - len(usadas)} descartadas por tener menos de {MIN_PALABRAS} palabras)")
print(f"{n_pal.sum()} palabras | {n_pal.min()}-{n_pal.max()} palabras por frase | "
      f"{len(usadas) - len({o for _, o, _ in items})} líneas repetidas en el corpus")
print(f"audio estimado: {sum(len(t.replace(' ', '')) for _, _, t in items) / CPS_SK001 / 3600:.1f} h")

# orden de proceso barajado con semilla: cualquier corte parcial es una muestra representativa
orden = list(range(len(items)))
np.random.RandomState(SEED).shuffle(orden)
if N_MAX:
    orden = orden[:N_MAX]

def semilla_de(id_):
    return (SEED + zlib.crc32(id_.encode())) % (2 ** 31)

device = torch.device("cuda")
tacotron, hp = C.cargar_tacotron(RUTA_TACOTRON, device, MAX_PASOS, STOP_THRESHOLD)
hifigan, h, denoiser = C.cargar_hifigan(RUTA_HIFIGAN, "/content/hifi-gan/config_v1.json", device, SEED)
print(f"modelos cargados | n_symbols={hp.n_symbols} | vocoder {h.sampling_rate} Hz")
qc_args = (FOCO_MIN, COBERTURA_MIN, (RATIO_DUR_MIN, RATIO_DUR_MAX))

## 7. Prueba corta — escúchala antes de seguir

Si suena mal aquí, suena mal en las 23 mil. Revisa también el gráfico de atención:
una **diagonal limpia** es buena señal; manchas, cortes o una línea que no llega al final
indican que el modelo no está alineando el texto.

In [ ]:
import IPython.display as ipd, matplotlib.pyplot as plt

for k in orden[:3]:
    id_, _, texto = items[k]
    y, q = C.sintetizar(texto, semilla_de(id_), tacotron, hifigan, denoiser, device,
                        MAX_PASOS, CPS_SK001, FUERZA_DENOISER)
    print(f"\n{id_} | {texto}")
    print(f"   estado: {C.estado_qc(q, *qc_args)} | {q}")
    if y is not None:
        ipd.display(ipd.Audio(y, rate=22050))

# atención de la última frase
from text import text_to_sequence
with torch.no_grad():
    C.fijar_semilla(semilla_de(id_))
    s = torch.LongTensor(text_to_sequence(texto + ";", ["basic_cleaners"]))[None].to(device)
    _, mel, _, al = tacotron.inference(s)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
ax[0].imshow(mel[0].cpu().numpy(), aspect="auto", origin="lower", cmap="inferno"); ax[0].set_title("mel")
ax[1].imshow(al[0].cpu().numpy().T, aspect="auto", origin="lower", cmap="inferno"); ax[1].set_title("atención")
ax[1].set_xlabel("paso del decoder"); ax[1].set_ylabel("posición en el texto")
plt.tight_layout(); plt.show()

## 8. Generación en lote

Escribe cada wav y su fila del manifest al instante. Si se corta, **vuelve a ejecutar
desde la celda 1**: retoma donde quedó. Las frases que fallan quedan anotadas con su
motivo en `estado` y no detienen la corrida.

In [ ]:
import csv, os, time, soundfile as sf, torchaudio
from tqdm.auto import tqdm

out = Path(SALIDA); out.mkdir(parents=True, exist_ok=True)
MANIFEST = out / "manifest.csv"
CAMPOS = ["id", "archivo", "texto", "texto_original", "n_palabras", "dur_s", "seed", "pasos",
          "toco_max", "foco", "cobertura_fin", "dur_esperada", "ratio_dur", "estado", "t_gen_s"]

# --- config de la corrida: todo lo necesario para reproducirla
cfg = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str))}
cfg.update(
    torch=torch.__version__, cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0),
    python=platform.python_version(), numpy=np.__version__, librosa=librosa.__version__,
    sha256_texto=C.sha256(RUTA_TEXTO), sha256_tacotron=C.sha256(RUTA_TACOTRON),
    sha256_hifigan=C.sha256(RUTA_HIFIGAN), tacotron_archivo=RUTA_TACOTRON.name,
    hifigan_archivo=RUTA_HIFIGAN.name, n_frases=len(items))
previo = out / "config.json"
if previo.exists():
    viejo = json.loads(previo.read_text())
    for k in ("sha256_texto", "sha256_tacotron", "sha256_hifigan", "SEED", "MAX_PALABRAS",
              "MIN_PALABRAS", "OUTPUT_SR", "STOP_THRESHOLD", "FUERZA_DENOISER"):
        if viejo.get(k) != cfg.get(k):
            raise SystemExit(f"'{k}' cambió respecto de la corrida anterior en {out} "
                             f"({viejo.get(k)} -> {cfg.get(k)}). Mezclarías audio de dos "
                             f"configuraciones: usa otra carpeta SALIDA.")
previo.write_text(json.dumps(cfg, indent=2, ensure_ascii=False))

import re
# --- reanudar: sanear el manifest de una corrida anterior
for basura in out.rglob("*.part"):          # escrituras cortadas a la mitad
    basura.unlink()
previas = {}
if MANIFEST.exists():
    with open(MANIFEST, encoding="utf-8", newline="") as f:
        for r in csv.DictReader(f):
            est = r.get("estado") or ""
            # fuera: filas truncadas por una desconexion, y errores que vale la pena reintentar.
            # Las marcas de QC se conservan: son deterministas y fallarian igual.
            if re.fullmatch(r"L[0-9]{5}_[0-9]{2}", r.get("id") or "") and est and not est.startswith("error"):
                previas[r["id"]] = {c: r.get(c, "") for c in CAMPOS}
tmp = MANIFEST.with_suffix(".tmp")
with open(tmp, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=CAMPOS); w.writeheader(); w.writerows(previas.values())
os.replace(tmp, MANIFEST)

pendientes = [k for k in orden if items[k][0] not in previas]
print(f"{len(previas)} ya hechas | {len(pendientes)} pendientes")

fm = open(MANIFEST, "a", newline="", encoding="utf-8")
wr = csv.DictWriter(fm, fieldnames=CAMPOS)

errores = 0
for k in tqdm(pendientes, desc="sintetizando"):
    id_, original, texto = items[k]
    seed = semilla_de(id_)
    t0 = time.time()
    fila = dict(id=id_, archivo="", texto=texto, texto_original=original,
                n_palabras=len(texto.split()), seed=seed)
    try:
        y, q = C.sintetizar(texto, seed, tacotron, hifigan, denoiser, device,
                            MAX_PASOS, CPS_SK001, FUERZA_DENOISER)
        fila.update({c: q.get(c) for c in CAMPOS if c in q})
        fila["estado"] = C.estado_qc(q, *qc_args)
        if y is not None:
            if OUTPUT_SR != 22050:
                y = torchaudio.functional.resample(torch.from_numpy(y), 22050, OUTPUT_SR).numpy()
            rel = f"{id_[:3]}/{id_}.wav"          # subcarpetas de ~1000: Drive se pone lento con más
            (out / id_[:3]).mkdir(exist_ok=True)
            tmp = out / f"{rel}.part"
            sf.write(tmp, y, OUTPUT_SR, subtype="PCM_16", format="WAV")
            os.replace(tmp, out / rel)                 # solo existe el wav si se escribió completo
            fila["archivo"], fila["dur_s"] = rel, round(len(y) / OUTPUT_SR, 3)
    except Exception as e:                              # una frase rota no tumba 23 mil
        errores += 1
        fila["estado"] = f"error: {type(e).__name__}: {e}"[:200]
        torch.cuda.empty_cache()
    fila["t_gen_s"] = round(time.time() - t0, 3)
    wr.writerow(fila)
    fm.flush()   # sin fsync a proposito: si se pierden filas, esas frases se regeneran identicas
fm.close()
print(f"listo | errores: {errores}")

## 9. Resumen

In [ ]:
import pandas as pd
m = pd.read_csv(Path(SALIDA) / "manifest.csv")
ok = m[m.estado == "ok"]
print(f"frases procesadas : {len(m)} de {len(items)}")
print(f"con audio         : {m.archivo.notna().sum()}  = {m.dur_s.sum() / 3600:.2f} h")
print(f"estado ok         : {len(ok)}  = {ok.dur_s.sum() / 3600:.2f} h  ({100 * len(ok) / max(len(m), 1):.1f}%)")
print(f"velocidad         : {m.dur_s.sum() / m.t_gen_s.sum():.1f}x tiempo real\n")
print("motivos marcados (una frase puede tener varios):")
est = m.estado.where(~m.estado.str.startswith("error"), "error")
motivos = est[est != "ok"].str.split(",").explode().str.strip().value_counts()
print(motivos.to_string() if len(motivos) else "   ninguno")
print(f"\nfoco de atención: mediana {m.foco.median():.2f} | p5 {m.foco.quantile(.05):.2f}"
      f"  <- calibra FOCO_MIN con esto")